Part 1: Data Preprocessing & Feature Engineering

In [168]:
import requests
from pathlib import Path
import polars as pl
import numpy as np
import pandas as pd
import pyarrow as pa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import(
    mean_absolute_error, mean_squared_error, r2_score, 
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [169]:
# Define URLs for required files
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Create data/raw directory if it doesn't exist
BASE_DIR = Path.cwd().resolve()
data_dir = BASE_DIR / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Defines File paths for downloaded data
taxi_path = data_dir / "yellow_tripdata_2024-01.parquet"
zone_path = data_dir / "taxi_zone_lookup.csv"

# Download Files and write to specified paths
def download_file(url, path):
    if path.exists():
        return
     
    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

download_file(taxi_url, taxi_path)
download_file(zone_url, zone_path)
print("\nFiles downloaded successfully!")


Files downloaded successfully!


In [170]:
# Load Taxi Data with Polars
df = pl.read_parquet(taxi_path)

# Load Zone Lookup Data with Polars
df_zones = pl.read_csv(zone_path)

# Remove rows with nulls
def remove_nulls(df):
    num_rows = df.height

    critical_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", 
                    "DOLocationID", "fare_amount"]
    
    df = df.drop_nulls(critical_columns)

    removed_nulls = num_rows - df.height
    return df, removed_nulls

# Filter out invalid trips tracking reasons for removal
def filter_trips(df):
    current_rows = df.height

    df = df.filter(pl.col("trip_distance") > 0)
    invalid_distance = current_rows - df.height
    current_rows = df.height

    df = df.filter(pl.col("fare_amount") > 0)
    negative_fare = current_rows - df.height
    current_rows = df.height

    df = df.filter(pl.col("fare_amount") <= 500)
    exceeding_max = current_rows - df.height

    return df, invalid_distance, negative_fare, exceeding_max

# Filter out trips with dropoff before pickup
def filter_time(df):
    num_rows = df.height

    df = df.filter(pl.col("tpep_dropoff_datetime") > pl.col("tpep_pickup_datetime"))

    removed_time = num_rows - df.height
    return df, removed_time

# Print summary of removals
def save_and_print(df, total_removed, removed_nulls, invalid_distance, negative_fare, exceeding_max, removed_time):
    print("\n=== Cleaned Dataset Summary ===")
    print(f"Total rows removed: {total_removed:,}")
    print(f"Removed null values: {removed_nulls:,}")
    print(f"Removed invalid distances: {invalid_distance:,}")
    print(f"Removed negative fares: {negative_fare:,}")
    print(f"Removed exceeding $500: {exceeding_max:,}")
    print(f"Removed invalid times: {removed_time:,}")

original_rows = df.height

df, removed_nulls = remove_nulls(df)
df, invalid_distance, negative_fare, exceeding_max = filter_trips(df)
df, removed_time = filter_time(df)

total_removed = original_rows - df.height

save_and_print(df, total_removed, removed_nulls, invalid_distance, negative_fare, exceeding_max, removed_time)


=== Cleaned Dataset Summary ===
Total rows removed: 95,052
Removed null values: 0
Removed invalid distances: 60,371
Removed negative fares: 34,539
Removed exceeding $500: 30
Removed invalid times: 112


In [171]:
# Filter to credit card payments only (payment_type == 1)
df = df.filter(pl.col("payment_type") == 1)
print(df.shape)

(2298347, 19)


In [172]:
# Feature Engineering - Temporal Features
df = df.with_columns([
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),

    # Adjusting weekday to have Monday=0, Sunday=6 by minusing 1
    (pl.col("tpep_pickup_datetime").dt.weekday() - 1).alias("pickup_day_of_week"),
])

df = df.with_columns(
    (pl.col("pickup_day_of_week") >= 5).alias("is_weekend")
)

In [173]:
# Feature Engineering - Trip Features
df = df.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 60)
    .alias("trip_duration_minutes"),

    pl.col("trip_distance").log1p().alias("log_trip_distance")
])

df = df.with_columns([
    (pl.when(pl.col("trip_duration_minutes") > 0)
    .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
    .otherwise(0)
    ).alias("trip_speed_mph")
])

In [174]:
# Feature Engineering - Fare Features
df = df.with_columns([
    # Ensuring we don't divide by zero for distance by using a conditional expression
    (pl.when(pl.col("trip_distance") > 0)
    .then(pl.col("fare_amount") / pl.col("trip_distance"))
    .otherwise(0)
    ).alias("fare_per_mile"),
    
    # Ensuring we don't divide by zero for duration by using a conditional expression
    (pl.when(pl.col("trip_duration_minutes") > 0)
    .then(pl.col("fare_amount") / pl.col("trip_duration_minutes"))
    .otherwise(0)
    ).alias("fare_per_minute")
])

In [175]:
# Feature Engineering - Zone Features
# Join to get pickup borough
df = df.join(
    df_zones.select(["LocationID", "Borough"])
    .rename({
        "LocationID": "PULocationID", 
        "Borough": "pickup_borough"
    }), 
    on="PULocationID", 
    how="left"
)

# Join to get dropoff borough
df = df.join(
    df_zones.select(["LocationID", "Borough"])
    .rename({
        "LocationID": "DOLocationID", 
        "Borough": "dropoff_borough"
    }), 
    on="DOLocationID", 
    how="left"
)

# Define the Encoder to be used for both pickup and dropoff boroughs
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# One-hot encode the pickup borough columns
encoded = encoder.fit_transform(df.select("pickup_borough").to_numpy())
encoded_pickup = pl.DataFrame(encoded, 
                              schema = encoder.get_feature_names_out(["pickup_borough"])
                              .tolist())
df = df.hstack(encoded_pickup)

# One-hot encode the dropoff borough columns
encoded = encoder.fit_transform(df.select("dropoff_borough").to_numpy())
encoded_dropoff = pl.DataFrame(encoded, 
                               schema = encoder.get_feature_names_out(["dropoff_borough"])
                               .tolist())
df = df.hstack(encoded_dropoff)

print(f"\nDataFrame shape after all feature engineering: {df.shape}")
print(f"Columns: {df.columns}")


DataFrame shape after all feature engineering: (2298347, 45)
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'log_trip_distance', 'trip_speed_mph', 'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unkn

In [176]:
# Target 1: tip_amount (already a column)
# Target 2: high_tip – 1 if tip_amount > 20% of fare_amount
df = df.with_columns([
    pl.when(pl.col("tip_amount") > (pl.col("fare_amount") * 0.20))
    .then(1)
    .otherwise(0)
    .alias("high_tip")
])

# Class distribution
high_tip_counts = df["high_tip"].value_counts().sort("high_tip")
print("\nhigh_tip class distribution:")
print(high_tip_counts)


high_tip class distribution:
shape: (2, 2)
┌──────────┬─────────┐
│ high_tip ┆ count   │
│ ---      ┆ ---     │
│ i32      ┆ u32     │
╞══════════╪═════════╡
│ 0        ┆ 553151  │
│ 1        ┆ 1745196 │
└──────────┴─────────┘


In [177]:
# Convert to Pandas for splitting, scaling and modeling
df = df.to_pandas()
print(df.shape)
print(f'Original Columns: {df.columns.tolist()}')

# Identify datetime columns for exclusion
datetime_cols = df.select_dtypes(include='datetime').columns.tolist()

# Identify just the pickup_borough and dropoff_borough columns for exclusion since we have already one-hot encoded them
borough_cols = ["pickup_borough", "dropoff_borough"]

df = df.drop(columns=datetime_cols + borough_cols)
print(df.shape)
print(f'Remaining Columns: {df.columns.tolist()}')

(2298347, 46)
Original Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'log_trip_distance', 'trip_speed_mph', 'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown', 'high_tip']
(2298347, 42)
Remaini

In [178]:
# Define RANDOM_STATE for reproducibility
RANDOM_STATE = 42

target_columns = ["tip_amount", "high_tip"]

X = df.drop(columns=target_columns)
y_reg = df["tip_amount"]    # regression target
y_clf = df["high_tip"]     # classification target

# Step 1: split off training set (70 %)
X_train, X_temp, y_reg_train, y_reg_temp, y_clf_train, y_clf_temp = train_test_split(
    X, y_reg, y_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# Step 2: split remaining into test (15 %) and validation (15 %)
X_test, X_val, y_reg_test, y_reg_val, y_clf_test, y_clf_val = train_test_split(
    X_temp, y_reg_temp, y_clf_temp,
    test_size=0.5,
    random_state=RANDOM_STATE,
    stratify=y_clf_temp
)

In [179]:
# Identify numeric and categorical features
numeric_features = [col for col in X.columns if X[col].dtype in ['int8', 'int32', 'int64', 'float64', 'bool']]
categorical_features = [col for col in X.columns if X[col].dtype == 'object']

print(f"\nNumeric features: {numeric_features}")
print(f"Categorical features: {categorical_features}")

# Define numeric transformer with imputation and scaling
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Define categorical transformer with imputation and one-hot encoding
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine transformers into a ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


Numeric features: ['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'log_trip_distance', 'trip_speed_mph', 'fare_per_mile', 'fare_per_minute', 'pickup_borough_Bronx', 'pickup_borough_Brooklyn', 'pickup_borough_EWR', 'pickup_borough_Manhattan', 'pickup_borough_N/A', 'pickup_borough_Queens', 'pickup_borough_Staten Island', 'pickup_borough_Unknown', 'dropoff_borough_Bronx', 'dropoff_borough_Brooklyn', 'dropoff_borough_EWR', 'dropoff_borough_Manhattan', 'dropoff_borough_N/A', 'dropoff_borough_Queens', 'dropoff_borough_Staten Island', 'dropoff_borough_Unknown']
Categorical features: ['store_and_fwd_flag']


In [180]:
# Print Number of samples in each split
print("=== Split Sizes ===")
print(f"  Train      : {X_train.shape[0]:>8,}  ({X_train.shape[0]/X.shape[0]:.1%})")
print(f"  Validation : {X_val.shape[0]:>8,}  ({X_val.shape[0]/X.shape[0]:.1%})")
print(f"  Test       : {X_test.shape[0]:>8,}  ({X_test.shape[0]/X.shape[0]:.1%})")

# Print high_tip class distribution in each split
print("\n=== high_tip Class Distribution ===")
print("\nTrain:")
print(y_clf_train.value_counts(normalize=True).sort_index())

print("\nValidation:")
print(y_clf_val.value_counts(normalize=True).sort_index())

print("\nTest:")
print(y_clf_test.value_counts(normalize=True).sort_index())

=== Split Sizes ===
  Train      : 1,608,842  (70.0%)
  Validation :  344,753  (15.0%)
  Test       :  344,752  (15.0%)

=== high_tip Class Distribution ===

Train:
high_tip
0    0.240673
1    0.759327
Name: proportion, dtype: float64

Validation:
high_tip
0    0.240674
1    0.759326
Name: proportion, dtype: float64

Test:
high_tip
0    0.240674
1    0.759326
Name: proportion, dtype: float64


In [181]:
# ===== Feature Summary =====
print("\n=== Feature Summary ===")

print(f"\nTotal features used for modeling: {len(X.columns)}")

print("\nNumeric Features:")
for col in numeric_features:
    print(f"  - {col} ({X[col].dtype})")

print("\nCategorical Features:")
for col in categorical_features:
    print(f"  - {col} ({X[col].dtype})")


# ===== Excluded Features =====
print("\n=== Excluded Features ===")

print("\nTarget Variables (excluded from X):")
for col in target_columns:
    print(f"  - {col} (target variable for prediction)")

print("\nDatetime Features:")
for col in datetime_cols:
    print(f"  - {col} (excluded because sklearn models cannot directly use datetime objects)")

print("\nCategorical Borough Features:")
for col in borough_cols:
    print(f"  - {col} (excluded because they are already one-hot encoded)")


=== Feature Summary ===

Total features used for modeling: 40

Numeric Features:
  - VendorID (int32)
  - passenger_count (int64)
  - trip_distance (float64)
  - RatecodeID (int64)
  - PULocationID (int32)
  - DOLocationID (int32)
  - payment_type (int64)
  - fare_amount (float64)
  - extra (float64)
  - mta_tax (float64)
  - tolls_amount (float64)
  - improvement_surcharge (float64)
  - total_amount (float64)
  - congestion_surcharge (float64)
  - Airport_fee (float64)
  - pickup_hour (int8)
  - pickup_day_of_week (int8)
  - is_weekend (bool)
  - trip_duration_minutes (float64)
  - log_trip_distance (float64)
  - trip_speed_mph (float64)
  - fare_per_mile (float64)
  - fare_per_minute (float64)
  - pickup_borough_Bronx (float64)
  - pickup_borough_Brooklyn (float64)
  - pickup_borough_EWR (float64)
  - pickup_borough_Manhattan (float64)
  - pickup_borough_N/A (float64)
  - pickup_borough_Queens (float64)
  - pickup_borough_Staten Island (float64)
  - pickup_borough_Unknown (float64)


Part 2: Model Training & Tuning

In [182]:
# Fit preprocessor on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform validation and test sets
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Preprocessing completed!")
print(f"Train shape: {X_train_processed.shape}")
print(f"Validation shape: {X_val_processed.shape}")
print(f"Test shape: {X_test_processed.shape}")

Preprocessing completed!
Train shape: (1608842, 41)
Validation shape: (344753, 41)
Test shape: (344752, 41)


In [183]:
# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_processed, y_reg_train)

# Random Forest Regressor (with reduced complexity for faster training)
rf_reg = RandomForestRegressor(
    n_estimators=30,
    max_depth=10,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_reg.fit(X_train_processed, y_reg_train)

print("Regression models trained successfully!")

Regression models trained successfully!


In [184]:
# ===== Regression Evaluation =====
def evaluate_regression(model, X, y_true, model_name):
    y_pred = model.predict(X)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name}")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R2  : {r2:.4f}")

# Evaluate models
evaluate_regression(lin_reg, X_val_processed, y_reg_val, "Linear Regression")
evaluate_regression(rf_reg, X_val_processed, y_reg_val, "Random Forest Regressor")


Linear Regression
MAE : 0.0750
RMSE: 0.2214
R2  : 0.9967

Random Forest Regressor
MAE : 0.9036
RMSE: 1.7902
R2  : 0.7857


In [ ]:
# Logistic Regression
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE
)
log_reg.fit(X_train_processed, y_clf_train)

# Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_clf.fit(X_train_processed, y_clf_train)

print("Classification models trained successfully!")

In [ ]:
# ===== Classification Evaluation =====
def evaluate_classification(model, X, y_true, model_name):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f"\n{model_name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC AUC  : {auc:.4f}")

# Evaluate models
evaluate_classification(log_reg, X_val_processed, y_clf_val, "Logistic Regression")
evaluate_classification(rf_clf, X_val_processed, y_clf_val, "Random Forest Classifier")